In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# データの列名を定義
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model year', 'origin', 'car name']

# データの読み込み
url = 'http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
df = pd.read_csv(url, names=column_names, sep=r'\s+', na_values='?')

# 欠損値の処理
df = df.dropna().reset_index(drop=True)

df

In [ ]:
# 特徴量のリストから'origin'を除外
features = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
            'acceleration', 'model year']
X = df[features].values

# 'origin'のデータを取得（プロット用）
origins = df['origin'].values

# 平均と標準偏差を計算
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)

# 標準化
X_std = (X - mean) / std

# 共分散行列の計算
cov_matrix = np.cov(X_std, rowvar=False)

# 固有値と固有ベクトルの計算
eigen_values, eigen_vectors = np.linalg.eigh(cov_matrix)

# 固有値の降順でソート
sorted_index = np.argsort(eigen_values)[::-1]
eigen_values = eigen_values[sorted_index]
eigen_vectors = eigen_vectors[:, sorted_index]

# 固有値を表示
print('Eigen values: ', eigen_values)

# 固有ベクトルをdfへ変換
ev = pd.DataFrame(eigen_vectors, columns=[f'PC{i+1}' for i in range(7)], index=features)

# 固有ベクトルを表示
print(ev)

In [ ]:
# 上位2つの主成分を選択
k = 2
eigenvector_subset = eigen_vectors[:, 0:k]

# データを主成分空間に変換
X_reduced = np.dot(X_std, eigenvector_subset)

# マーカーの定義（原産国ごとに異なる形状）
markers = {1: 'circle', 2: 'square', 3: 'triangle-up'}
place = {1: 'US', 2: 'Europe', 3: 'Japan'}

# プロットの設定
fig = go.Figure()

# 原産国ごとにデータをプロット
for origin_value, marker in markers.items():
    # originがorigin_valueのインデックスを取得
    indices = np.where(origins == origin_value)[0]
    # データを取得してプロット
    fig.add_trace(go.Scatter(
        x=X_reduced[indices, 0], y=X_reduced[indices, 1],
        mode='markers', marker=dict(symbol=marker, size=8),
        name=place[origin_value],
        customdata=df.loc[indices, ['car name'] + features].to_numpy(),
        hovertemplate=(
            '<b>%{customdata[0]}</b><br>'
            'mpg: %{customdata[1]}<br>'
            'cylinders: %{customdata[2]}<br>'
            'displacement: %{customdata[3]}<br>'
            'horsepower: %{customdata[4]}<br>'
            'weight: %{customdata[5]}<br>'
            'acceleration: %{customdata[6]}<br>'
            'model year: %{customdata[7]}<br>'
            f'origin: {place[origin_value]}<extra></extra>'
        )
    ))

# 主成分負荷量（ロードイング）の取得
loading_scores = eigen_vectors[:, :k] * np.sqrt(eigen_values[:k])

# 特徴量名
feature_names = features

# ロードイングのスケーリング（視覚的にわかりやすくするため）
scaling_factor = 1  # 適宜調整可能

# ロードイングをプロット
for i in range(len(feature_names)):
    fig.add_annotation(
        x=loading_scores[i, 0]*scaling_factor, y=loading_scores[i, 1]*scaling_factor,
        ax=0, ay=0, xref='x', yref='y', axref='x', ayref='y',
        showarrow=True, arrowcolor='red', arrowwidth=2, arrowhead=3, opacity=0.7, text=''
    )
    fig.add_annotation(
        x=loading_scores[i, 0]*scaling_factor*1.5,
        y=loading_scores[i, 1]*scaling_factor*1.5,
        text=feature_names[i], showarrow=False, font=dict(color='red', size=12)
    )

# 軸ラベルとタイトル
fig.update_layout(xaxis_title='Principal Component 1', yaxis_title='Principal Component 2',
                  width=900, height=900)

# 凡例とグリッド
fig.update_xaxes(showgrid=True)
fig.update_yaxes(showgrid=True)

# プロットの表示
fig.show()

In [ ]:
# 寄与率の計算
explained_variance_ratio = eigen_values / np.sum(eigen_values)
cumulative_variance_ratio = np.cumsum(explained_variance_ratio)

# 寄与率を表示
for i, (evr, cv) in enumerate(zip(explained_variance_ratio, cumulative_variance_ratio)):
    print(f'Principal Component {i+1}: Explained Variance Ratio = {evr:.4f}, Cumulative = {cv:.4f}')

In [ ]:
# ロードイングをデータフレームにまとめる
loading_matrix = pd.DataFrame(loading_scores, columns=[f'PC{i+1}' for i in range(k)], index=feature_names)

# ロードイングを表示
print(loading_matrix)